# 06 - Task-vector and audit analysis

In [ ]:
import exp_common as E
import pandas as pd, json, torch
import matplotlib.pyplot as plt

RUN_SEEDS = [42, 123, 777]
all_rows = []
summary_rows = []

for seed in RUN_SEEDS:
    logs = E.ADAPTERS_DIR / "logs" / f"seed_{seed}" / "adapter"
    qa = E.ADAPTERS_DIR / "qa" / f"seed_{seed}" / "adapter"
    if not (logs / "adapter_model.safetensors").exists() or not (qa / "adapter_model.safetensors").exists():
        raise FileNotFoundError(f"Missing base-task adapters for seed {seed}")
    prefix = E.ANALYSIS_DIR / f"task_vectors_seed_{seed}"
    df = E.adapter_vector_analysis(logs, qa, prefix)
    df["seed"] = seed
    all_rows.append(df)
    obj = json.loads((E.ANALYSIS_DIR / f"task_vectors_seed_{seed}_summary.json").read_text())
    summary_rows.append({
        "seed": seed,
        "global_cosine_A": obj.get("global_cosine_A"),
        "global_cosine_B": obj.get("global_cosine_B"),
        "global_cosine_delta_W": obj.get("global_cosine_delta_W"),
        "global_conflict_frac_delta_W": obj.get("global_conflict_frac_delta_W"),
        "logs_factor_l2": obj["a_factor"].get("l2"),
        "qa_factor_l2": obj["b_factor"].get("l2"),
        "logs_delta_W_l2": obj["a_delta_W"].get("l2"),
        "qa_delta_W_l2": obj["b_delta_W"].get("l2"),
    })

vectors_df = pd.concat(all_rows, ignore_index=True)
pd.DataFrame(summary_rows).to_csv(E.ANALYSIS_DIR / "task_vectors_summary.csv", index=False)
vectors_df.to_csv(E.ANALYSIS_DIR / "task_vectors_all_per_tensor.csv", index=False)

for col in ["cosine", "conflict_frac", "a_l2", "b_l2"]:
    if col in vectors_df:
        plt.figure(figsize=(7,4)); vectors_df[col].dropna().hist(bins=50); plt.title(col); plt.tight_layout(); plt.savefig(E.ANALYSIS_DIR / f"{col}_hist.png", dpi=160); plt.show()

print(pd.DataFrame(summary_rows))

## Per-vector audit

In [ ]:
audit_rows = []
for seed in RUN_SEEDS:
    logs = E.ADAPTERS_DIR / "logs" / f"seed_{seed}" / "adapter"
    qa = E.ADAPTERS_DIR / "qa" / f"seed_{seed}" / "adapter"
    audit = E.compute_audit(logs, qa)
    audit["seed"] = seed
    audit["task"] = "QA relative to Logs"
    audit_rows.append(audit)

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(E.ANALYSIS_DIR / "audit_base_task_pairs.csv", index=False)
audit_df